In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
cd /content/drive/MyDrive/ALS-Model-GPU

/content/drive/MyDrive/ALS-Model-GPU


In [4]:
!pip install pymilvus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.2/315.2 kB 10.0 MB/s eta 0:00:00


In [5]:
# from pipeline.run_pipeline import main

In [6]:
# del main

In [7]:
# import sys

# # Temporarily save original sys.argv
# original_argv = sys.argv[:]

# # Set sys.argv to simulate command line execution for argparse
# sys.argv = ['run_pipeline.py', '--config', 'config/config.yaml']

# try:
#     main()
# finally:
#     # Restore original sys.argv to avoid interfering with other parts of the notebook
#     sys.argv = original_argv

In [8]:
# %cd /content/drive/MyDrive/ALS-Model-GPU
# !python3 -m pipeline.run_pipeline --config config/config.yaml --from bridge

In [9]:
# !python3 -m pipeline.run_pipeline --config config/config.yaml --only infer

In [10]:
# !python3 -m pipeline.run_pipeline --config config/config.yaml --store faiss --from index


In [11]:
# !pip install faiss-cpu

In [12]:
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 14.3 MB/s eta 0:00:00


In [ ]:
!python3 -m pipeline.run_pipeline --config config/config.yaml --store faiss --only alu



ALU Cross-Domain Recommendation Pipeline
  Config       : config/config.yaml
  Vector store : faiss
  Stages       : alu
  Scale        : 8.6M users | 52K radio | 300K podcasts


  STAGE: ALU
[Trainer] Device: cuda:0
[Dataset] Loading feedback from data/feedback_data.pkl
[Dataset] Building user positive sets for negative sampling...
[Dataset] 1,371,514 users | 50,621 items | 8,443,535 interactions
[Trainer] Users: 1,371,514 | Items: 50,621
/content/drive/MyDrive/ALS-Model-GPU/training/trainer.py:143: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg["model"]["use_fp16"])
[Trainer] Loading checkpoint from checkpoints/alu/alu_best.pt
  ✓ Resumed from epoch 4 | Best val_loss: 0.4691
/content/drive/MyDrive/ALS-Model-GPU/training/trainer.py:171: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocas

In [5]:
!python3 -m pipeline.run_pipeline --config config/config.yaml --store faiss --only bridge



ALU Cross-Domain Recommendation Pipeline
  Config       : config/config.yaml
  Vector store : faiss
  Stages       : bridge
  Scale        : 8.6M users | 52K radio | 300K podcasts


  STAGE: BRIDGE
[Bridge Trainer] Device: cpu
[Bridge Trainer] Loading ALU from checkpoints/alu/alu_best.pt
[Bridge Trainer] Computing radio item latents...
/content/drive/MyDrive/ALS-Model-GPU/training/bridge_trainer.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=cfg["model"]["use_fp16"]):
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
  Radio latents: torch.Size([50621, 128]) | Podcast latents: torch.Size([207716, 128])
[Bridge] Building anchor pairs (threshold=0.75)
         Radio: 50,621 | Podcasts: 207,716
  processed 512/50,621 radio items | pairs foun

In [6]:
# !pip install faiss-cpu

In [7]:
import torch

In [8]:
torch.cuda.is_available()

False

In [9]:
!python3 -m pipeline.run_pipeline --config config/config.yaml --store faiss --only index


ALU Cross-Domain Recommendation Pipeline
  Config       : config/config.yaml
  Vector store : faiss
  Stages       : index
  Scale        : 8.6M users | 52K radio | 300K podcasts


  STAGE: INDEX  [backend: faiss]
[FAISS] Building podcast index on cpu
[FAISS] Computing podcast latents...
/content/drive/MyDrive/ALS-Model-GPU/inference/faiss_index.py:84: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=cfg["model"]["use_fp16"]):
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
  50,000/207,716 podcasts encoded
  100,000/207,716 podcasts encoded
  150,000/207,716 podcasts encoded
  200,000/207,716 podcasts encoded
  207,716/207,716 podcasts encoded
[FAISS] Podcast latents: (207716, 128)
[FAISS] Saved podcast latents → data/podcast_latents.npy
[FAISS

In [10]:
!python3 -m pipeline.run_pipeline --config config/config.yaml --store faiss --from infer



ALU Cross-Domain Recommendation Pipeline
  Config       : config/config.yaml
  Vector store : faiss
  Stages       : infer
  Scale        : 8.6M users | 52K radio | 300K podcasts


  STAGE: INFER  [backend: faiss]
[Infer] Device: cpu
[Infer] Loading FAISS index from checkpoints/podcast_faiss.index
[Infer] FAISS index on GPU: 207,716 podcasts
[Infer] 1,371,514 users | chunk=10,000 | top_k=50 | 138 chunks

[Chunk 1/138] users 0–10,000
/content/drive/MyDrive/ALS-Model-GPU/inference/batch_infer.py:141: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=cfg["model"]["use_fp16"]):
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
  Written 10,000 rows → reco_chunk_000.parquet (7s | ETA 16.0min)

[Chunk 2/138] users 10,000–20,000
  Written 10,000 rows → re

In [ ]:
!python3 recommend.py --user-id 100

^C


In [ ]:
!python3 recommend.py --user-id 10000 --top-k 10 --format table

user_id: 10000
source_file: outputs/recommendations/reco_chunk_018.parquet

  1. 268539_pd  score=0.9216
  2. 256819_pd  score=0.9011
  3. 256918_pd  score=0.9000
  4. 257776_pd  score=0.8852
  5. 268342_pd  score=0.8828
  6. 252493_pd  score=0.8787
  7. 237009_pd  score=0.8748
  8. 265121_pd  score=0.8742
  9. 232742_pd  score=0.8696
 10. 268340_pd  score=0.8694


*may-5*

In [ ]:
!python3 recommend.py --user-id 10000 --top-k 10 --format table

user_id: 10000
source_file: outputs/recommendations/reco_chunk_018.parquet

  1. 256819_pd  score=0.8875
  2. 237009_pd  score=0.8791
  3. 257371_pd  score=0.8712
  4. 238478_pd  score=0.8687
  5. 256918_pd  score=0.8675
  6. 252493_pd  score=0.8575
  7. 256802_pd  score=0.8526
  8. 257381_pd  score=0.8513
  9. 257383_pd  score=0.8466
 10. 257370_pd  score=0.8398


*may-6*

In [11]:
!python3 recommend.py --user-id 10000 --top-k 10 --format table

user_id: 10000
source_file: outputs/recommendations/reco_chunk_018.parquet

  1. 238478_pd  score=0.9133
  2. 256819_pd  score=0.9131
  3. 256182_pd  score=0.9084
  4. 257371_pd  score=0.9069
  5. 268539_pd  score=0.9068
  6. 256802_pd  score=0.9059
  7. 237009_pd  score=0.8978
  8. 257370_pd  score=0.8948
  9. 256933_pd  score=0.8923
 10. 259990_pd  score=0.8900
